In [ ]:
# general
import sys, os
import time
from os.path import join
from os import path
from importlib import reload
from getpass import getuser
from glob import glob
from tqdm.auto import tqdm
from multiprocessing import Pool
import shutil 
import gc


# Data
import xarray as xr
import h5py
import numpy as np
import imageio
from nexusformat.nexus import *
from PIL import Image
from scipy.optimize import curve_fit

# Plotting
import matplotlib.pyplot as plt

# skimage
import skimage.morphology

# scipy
from scipy.ndimage import gaussian_filter
from scipy import stats
import scipy
from scipy.interpolate import griddata

# pyFAI
import pyFAI
from pyFAI.azimuthalIntegrator import AzimuthalIntegrator
from pyFAI.detectors import Detector

# Self-written libraries
sys.path.append(join(os.getcwd(), "library"))
import helper_functions as helper
import mask_lib
import interactive
from interactive import cimshow
import phase_retrieval_core as PhR
import Phase_Retrieval_old_but_gold as phr_gold

plt.rcParams["figure.constrained_layout.use"] = True  # replaces plt.tight_layout

# Is there a GPU?
try:
    # Cupy
    import cupy as cp
    import cupyx as cpx

    GPU = True

    print("GPU available")

    # Self-written library
    import CCI_core_cupy as cci
except:
    GPU = False
    import CCI_core as cci

    print("GPU unavailable")
parula_map = cci.parula_map()

# interactive plotting
import ipywidgets

%matplotlib widget

# Auto formatting of cells
#%load_ext jupyter_black

In [ ]:
facility = "MAXIV" # Options: "SwissFEL", "MAXI"
BEAMTIMEID = 2026031908 # Proposal number
data_fname_prefix = "2603_softimax"
USER = getuser()

# Facility specific loading functions
if facility == "PETRA":
    import PETRA_MaxP04_loading as loading
elif facility == "MAXI":
    import MAXI_loading as loading
elif facility == "SwissFEL":
    from sfdata import SFDataFiles, SFScanInfo, SFProcFile
    import Swiss_FEL_Loading as loading

    # Number or jobs for analysis
    NR_JOBS = 32
elif facility == "MAXIV":
    import MAXI_loading as loading

BASEFOLDER = "/data/export/sgaebel/26_03_MAXIV"
DATAFOLDER = join(BASEFOLDER,"raw")
folder_general = join(BASEFOLDER, "process")
print("Raw Datafolder is: %s"%DATAFOLDER)

# Load dictionary for keys etc
mnemonics = loading.load_mnemonics()

In [ ]:
#im_id_set = [5292, 5294, 5308, 5314,5310,5316,5312, 5318  ]
#topo_id_set = [5294, 5292, 5314, 5308, 5316, 5310,5318 , 5312]
#im_id_set = [5208, 5210, 5222 ,5216,5220, 5214, 5218, 5212 ]
#topo_id_set = [5210, 5208, 5216, 5222, 5214, 5220,5212 , 5218]
#im_id_set = [5224, 5226, 5238, 5232, 5236, 5230, 5234, 5228 ]
#topo_id_set = [5226, 5224, 5232, 5238, 5230, 5236,5228 , 5234]
#im_id_set = [5320, 5322, 5308, 5314, 5310, 5316, 5312, 5318  ]
#topo_id_set = [5322, 5320, 5314, 5308, 5316, 5310, 5318 , 5312]
#im_id_set = [5376, 5378, 5390, 5384, 5388, 5382, 5386, 5380 ]
#topo_id_set = [5378, 5376, 5384, 5390, 5382, 5388, 5380 , 5386]
#im_id_set = [5398, 5396, 5410, 5404, 5408, 5402, 5406, 5400]
#topo_id_set = [5396, 5398, 5404, 5410, 5402, 5408, 5400 , 5406]
#im_id_set = [5083, 5085, 5087, 5093, 5089, 5095, 5091, 5097] #np.arange(4266, 4291+1)
#topo_id_set = [5085, 5083, 5093, 5087, 5095, 5089, 5097, 5091]#np.arange(4266, 4291+1) #4266*np.ones(len(im_id_set), dtype=int)
#dark_id_im_set = [5084, 5086, 5088, 5094, 5092, 5094, 5096, 5098]#* np.ones(len(im_id_set), dtype=int)
#im_id_set = [5376, 5378, 5380, 5386, 5382,5388, 5384,5390  ]
#topo_id_set = [5378, 5376, 5386, 5380,5388, 5382, 5390, 5384 ]
im_id_set = [6188, 6190,6203, 6197, 6201, 6195, 6199, 6193  ]#np.arange(7096, 7114+1)[::2]
topo_id_set = [6190, 6188, 6197, 6203, 6195, 6201, 6193 , 6199]
#im_id_set = [6309, 6311, 6313, 6323, 6315, 6321, 6317, 6319  ]#np.arange(5712, 5676+1)[::2]
#topo_id_set = [ 6311, 6309, 6323, 6313, 6321, 6315, 6319, 6317  ]#* np.ones(len(im_id_set), dtype=int)#np.arange(4266, 4291+1) #4266*np.ones(len(im_id_set), dtype=int)
#im_id_set = [6398, 6400, 6404, 6410, 6402, 6412, 6406, 6408]
#topo_id_set = [6400, 6398, 6410, 6404, 6412, 6402, 6408, 6406]
#im_id_set = [6475, 6477, 6479, 6489, 6481, 6487, 6483, 6485]
#topo_id_set = [6477, 6475, 6489, 6479, 6487, 6481, 6485, 6483]
#im_id_set = [6556, 6550, 6548, 6558, 6554, 6552,6560, 6562 ]#np.arange(6533, 6543+1)
#topo_id_set = [6550, 6556, 6558, 6548, 6552, 6554, 6562, 6560]
#im_id_set = [6572,6566, 6564,6574, 6570, 6568, 6578, 6576 ]
#topo_id_set = [6566,6572, 6574,6564, 6568, 6570, 6576, 6578 ]
#im_id_set = [6584, 6586, 6588, 6598, 6590, 6596, 6592, 6594]
#topo_id_set = [6586, 6584, 6598, 6588, 6596, 6590, 6594, 6592]
#im_id_set = [6989, 6991,6993, 7003, 6995, 7001, 6997, 6999 ]
#topo_id_set = [6991,6989, 7003, 6993, 7001, 6995, 6999, 6997 ]

p_pc_list = []
n_pc_list = []
sum_list = []
for im_id, topo_id in zip(im_id_set, topo_id_set):
    fname_h5 = join(folder_general, "Recon_ImId_%04d_RefId_%s_cdi_stack_%s.h5" % (im_id, topo_id, USER))
    with h5py.File(fname_h5, "r") as f:
        sum_list.append( np.sum(f["image"][()]))
        p_pc_list.append(f["p_pc"][()])
        n_pc_list.append(f["n_pc"][()])
        supportmask = f["support_mask"][()]
p_pc_list = np.array(p_pc_list)[:,710:865, 775:942] 
n_pc_list = np.array(n_pc_list)[:,710:865, 775:942] 
supportmask = supportmask[710:865, 775:942]  #[760:925, 678:852]    5083, 5292, 5376, 5208, 5320, 5376, 5398; [:, 610:825, 720:925]  6309; [611:821, 707:916], 6398; [:, 713:882, 760:933], 6475; [:, 818:916,  910:1003], 6556
#[:, 574:814, 665:903], 6572; [:,848:961, 799:909] 5224;
#angles = [ 0, 90, -30, 60, -45, 45, -60, 30] #5208
#angles = [ 0, 90, -30, 60, -45, 45, -60, 30] #5224
#angles = [ 0, 90, -30, 60, -45, 45, -60, 30] #5320
#angles = [ 0, 90, -30, 60, -45, 45, -60, 30] #5376
#angles = [ 0, 90, -30, 60, -45, 45, -60, 30] #5398
#angles = [ 0, 90, -30, 60, -45, 45, -60, 30] #5292
#angles = [0, 90, 30, -60, 45, -45, 60, -30] #5083
#angles = [90, 0, 30, -60, 45, -45, 60, -30] #5376
angles = [ 0, 90, -30, 60, -45, 45, -60, 30] #6188
#angles = [0, 90, -30, 60, 30, -60, -45, 45] #6309
#angles = [0, 90, 30, -60, -30, 60, -45, 45] #6398
#angles = [0, 90, -30, 60, 30, -60, -45, 45] #6475
#angles = [45, -45, -30, 60, 30, -60, 0, 90] #6556
#angles = [45, -45, -30, 60, 30, -60, 0, 90] #6572
#angles = [0, 90, -30, 60, -45, 45, -60, 30] #6989
data = np.real((n_pc_list-p_pc_list))*supportmask

In [ ]:
cimshow(data, cmap="gray")

In [ ]:
def cos_f(x, a, b, c):
    return a * np.cos( x + b)**2 + c

rad = np.deg2rad(angles)
y = data[:,66,46]
popt, cov = curve_fit(cos_f, rad, y, p0=(1e5, np.pi/2, -110), bounds=np.array([(0, 1e8), (0, np.pi), (-1000000, 1000000)]).T)

rad_2 = np.linspace(-1, 1.5, 100)

plt.figure()
plt.plot(np.rad2deg(rad), y, ".")
plt.plot(np.rad2deg(rad_2), cos_f(rad_2, *popt), ".")

In [ ]:
from multiprocessing import Pool, cpu_count
from scipy.optimize import curve_fit

def worker(args):
    ix, iy, data, rad = args
    y = data[:, ix, iy]

    try:
        #popt, cov = curve_fit(cos_f, rad, y, p0=(15000, 1, 1000), bounds=np.array([(0, 1e8), (0, np.pi), (-1000000, 1000000)]).T)
        popt, cov = curve_fit(cos_f, rad, y, p0=(1e5, np.pi/2, -110), bounds=np.array([(0, 1e8), (0, np.pi), (-1000000, 1000000)]).T)
        #if np.any(popt/np.diag(cov)> 100):
          # return ix, iy, np.array([np.nan, np.nan, np.nan])
        return ix, iy, popt
    except Exception:
        return ix, iy, np.array([np.nan, np.nan, np.nan])

if __name__ == "__main__":
    tasks = [
        (ix, iy, data, rad)
        for ix in range(data.shape[1])
        for iy in range(data.shape[2])
    ]

    fit = [[None for _ in range(data.shape[2])] for _ in range(data.shape[1])]

    with Pool(cpu_count()) as pool:
        for ix, iy, result in tqdm(pool.imap(worker, tasks), total=len(tasks)):
            fit[ix][iy] = result

In [ ]:
cimshow(np.array(fit)[:,:,1], cmap="twilight_shifted")
cimshow(np.array(fit)[:,:,0], cmap="twilight_shifted")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb

Dat = np.array(fit)
amp = Dat[:, :, 0]
phase = Dat[:, :, 1]

# Phase normalisieren auf [0, 1]
#phase_norm = (phase - np.nanmin(phase)) / (np.nanmax(phase) - np.nanmin(phase))

# Alternativ besser bei echter Phase:
phase_norm = (phase % (2*np.pi)) / (2*np.pi)*2

# Amplitude normalisieren
amp_norm = amp / np.nanmax(np.abs(amp))

# HSV zusammensetzen
hsv = np.zeros((*amp.shape, 3))
hsv[..., 0] = phase_norm        # Hue
hsv[..., 1] = 1.0               # Saturation
hsv[..., 2] = amp_norm          # Value

rgb = hsv_to_rgb(hsv)

plt.figure()
plt.imshow(rgb)
plt.title(f"{im_id_set[0]}: Phase (Hue) + Amplitude (Value)")
plt.show()


fname = "/"+join(*fname_h5.split("/")[:-1])+"/XMLD_map_"+fname_h5.split("/")[-1][11:15]+".h5"
with h5py.File(fname, "w") as f:
    f.create_dataset("Amplitude", data = Dat[:, :, 0], chunks=True)
    f.create_dataset("phase", data = Dat[:, :, 1], chunks=True)
    f.create_dataset("data", data=data, chunks=True)
    f.create_dataset("angles", data=angles, chunks=True)
    f.create_dataset("topo_id", data=topo_id_set, chunks=True)
    f.create_dataset("im_id", data=im_id_set, chunks=True)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb

# --- Daten wie bei dir ---
Dat = np.array(fit)
amp = Dat[:, :, 0]
phase = Dat[:, :, 1]

phase_norm = (phase % (2*np.pi)) / (2*np.pi)*2
#phase_norm = (phase - np.nanmin(phase)) / (np.nanmax(phase) - np.nanmin(phase))
amp_norm = amp / np.nanmax(np.abs(amp))

hsv = np.zeros((*amp.shape, 3))
hsv[..., 0] = phase_norm
hsv[..., 1] = 1.0
hsv[..., 2] = amp_norm

rgb = hsv_to_rgb(hsv)

# --- Plot Hauptbild ---
fig, ax = plt.subplots(figsize=(6, 6))
im = ax.imshow(rgb)
ax.set_title("Phase (Hue) + Amplitude (Value)")
ax.axis('off')

# --- Kreis-Legende erzeugen ---
N = 300
x = np.linspace(-1, 1, N)
y = np.linspace(-1, 1, N)
X, Y = np.meshgrid(x, y)

R = np.sqrt(X**2 + Y**2)
Phi = np.arctan2(Y, X)

mask = R <= 1

phase_leg = (Phi % (2*np.pi)) / (2*np.pi)
amp_leg = np.clip(R, 0, 1)

hsv_leg = np.zeros((N, N, 3))
hsv_leg[..., 0] = phase_leg
hsv_leg[..., 1] = 1.0
hsv_leg[..., 2] = amp_leg

rgb_leg = hsv_to_rgb(hsv_leg)

# außerhalb des Kreises weiß setzen
rgb_leg[~mask] = 1.0

# --- Legende einfügen ---
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

axins = inset_axes(ax, width="30%", height="30%", loc='lower right')
axins.imshow(rgb_leg, extent=(-1,1,-1,1))
axins.set_xticks([])
axins.set_yticks([])
axins.set_title("Legend", fontsize=8)

plt.show()

In [ ]:


im_id_set = [7033, 7039, 7041, 7047]
topo_id_set = [7035, 7037, 7043, 7045]
p_pc_list = []
n_pc_list = []
for im_id, topo_id in zip(im_id_set, topo_id_set):
    fname_h5 = join(folder_general, "Recon_ImId_%04d_RefId_%s_cdi_stack_%s.h5" % (im_id, topo_id, USER))
    with h5py.File(fname_h5, "r") as f:
        p_pc_list.append(f["p_pc"][()])
        n_pc_list.append(f["n_pc"][()])
        supportmask = f["support_mask"][()]

In [ ]:
data = (p_pc_list[0]+n_pc_list[0])/(p_pc_list[1]+n_pc_list[1])*supportmask
data_1 = (p_pc_list[1]-n_pc_list[1])*supportmask

In [ ]:
cimshow(data, cmap="gray")

In [ ]:
cimshow(data, cmap="gray")

In [ ]:
cimshow(data_1, cmap="gray")